# Interview Scoring System

This notebook evaluates interview answers with Google Gemini, validates scores, and exports the results to `scored_answers.csv`.

Place a CSV answer file, a JSON scoring-rules file, and a `.key` file containing `GEMINI_API_KEY` in the working directory before running the execution cells.

## 1. Imports and configuration

In [ ]:
import json
import logging
import os
from pathlib import Path

import pandas as pd
from google import genai
from google.genai import types

import guardrails
import security

logger = logging.getLogger("agent")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    file_handler = logging.FileHandler("agent.log", encoding="utf-8")
    console_handler = logging.StreamHandler()
    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

logger.info("Startup working directory: %s", Path.cwd().resolve())

## 2. Load the API key and input files

In [ ]:
def try_load_api_key_from_key_file() -> str:
    key_files = sorted(Path(".").glob("*.key"))
    for key_file in key_files:
        try:
            key_value = key_file.read_text(encoding="utf-8").strip()
            if key_value:
                os.environ["GEMINI_API_KEY"] = key_value
                logger.info("Loaded GEMINI_API_KEY from %s", key_file.name)
                return key_value
        except OSError as error:
            logger.warning("Failed to read %s: %s", key_file.name, error)
    return ""

def load_the_json():
    json_rule_files = sorted(Path(".").glob("*.json"))
    if not json_rule_files:
        logger.error("No JSON rule files found.")
        return None
    if len(json_rule_files) > 1:
        logger.warning("Multiple JSON files found. Using %s", json_rule_files[0].name)
    with json_rule_files[0].open(encoding="utf-8") as file:
        return json.load(file)

def load_the_csv():
    csv_files = sorted(Path(".").glob("*.csv"))
    if not csv_files:
        logger.error("No CSV files found.")
        return None
    if len(csv_files) > 1:
        logger.warning("Multiple CSV files found. Using %s", csv_files[0].name)
    return pd.read_csv(csv_files[0])

api_key = os.getenv("GEMINI_API_KEY", "").strip() or try_load_api_key_from_key_file()
if not api_key:
    raise RuntimeError("GEMINI_API_KEY is not set. Set it in the environment or a .key file.")

client = genai.Client(api_key=api_key)
df = load_the_csv()
if df is None or df.empty:
    raise RuntimeError("No CSV file to process.")
df.columns = [str(column).strip(' \"') for column in df.columns]

scoring_rules = load_the_json()
if scoring_rules is None:
    raise RuntimeError("No JSON rule file to process.")

rules_dict = {
    question_key: rule_data
    for question_object in scoring_rules["questions"]
    for question_key, rule_data in question_object.items()
}

with Path("context.md").open(encoding="utf-8") as file:
    context_text = file.read()
with Path("specification.md").open(encoding="utf-8") as file:
    specification_text = file.read()
system_prompt = f"{context_text}\n\n{specification_text}"

print(f"Loaded {len(df)} answer rows and {len(rules_dict)} scoring rules.")

## 3. Agent tools and response schema

In [ ]:
def range_and_type_checker(score: float, range_min: float = 0.0, range_max: float = 10.0) -> str:
    try:
        if not isinstance(score, (int, float)):
            return "[ERROR] The score is not a number; please try again."
        if not range_min <= score <= range_max:
            return f"[ERROR] The score is not within the specified range ({range_min}, {range_max}); please try again."
        return "[INFO] The score is valid."
    except Exception as error:
        return f"[ERROR] An unexpected error occurred: {error}"

def confidentality_enhance(current_score: float, confidence: float, reasoning: str) -> str:
    if confidence < 0.5:
        return (
            f"[WARNING] Confidence is low ({confidence}); initiate a guided Self-Refine loop. "
            f"Reasoning: {reasoning}. Re-read the answer, then provide a new score and reasoning."
        )
    return f"[INFO] Confidence is sufficient ({confidence}); no Self-Refine loop needed. Reasoning: {reasoning}."

tool_config = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="range_and_type_checker",
        description="Validate that a score is numeric and within the requested range.",
        parameters={
            "type": "object",
            "properties": {"score": {"type": "number"}},
            "required": ["score"],
        },
    ),
    types.FunctionDeclaration(
        name="confidentality_enhance",
        description="Start guided self-refinement when confidence is below 0.5.",
        parameters={
            "type": "object",
            "properties": {
                "current_score": {"type": "number"},
                "confidence": {"type": "number"},
                "reasoning": {"type": "string"},
            },
            "required": ["current_score", "confidence", "reasoning"],
        },
    ),
])

final_output_schema = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "score": types.Schema(type=types.Type.NUMBER),
        "confidence": types.Schema(type=types.Type.NUMBER),
        "reasoning": types.Schema(type=types.Type.STRING),
    },
    required=["score", "confidence", "reasoning"],
)

## 4. Agent loop

In [ ]:
def agent_loop(
    llm_model,
    current_question,
    rules,
    range_min,
    range_max,
    answer_text,
    maximum_attempts=20,
):
    user_prompt = (
        f"Question: {current_question}\nRules: {rules}\n"
        f"Answer to evaluate: {answer_text}"
    )
    messages = [types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)])]
    attempts = 0

    while True:
        try:
            response = client.models.generate_content(
                model=llm_model,
                contents=messages,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    tools=[tool_config],
                    response_mime_type="application/json",
                    response_schema=final_output_schema,
                    temperature=0.1,
                ),
            )
            attempts = 0
        except Exception as error:
            attempts += 1
            logger.exception("Model call failed (%s/%s): %s", attempts, maximum_attempts, error)
            if attempts >= maximum_attempts:
                return None
            continue

        if response.text:
            return response.text

        if response.function_calls:
            messages.append(response.candidates[0].content)
            for call in response.function_calls:
                if call.name == "range_and_type_checker":
                    result = range_and_type_checker(**call.args, range_min=range_min, range_max=range_max)
                elif call.name == "confidentality_enhance":
                    result = confidentality_enhance(**call.args)
                else:
                    result = f"Unknown tool: {call.name}"
                messages.append(types.Content(
                    role="tool",
                    parts=[types.Part.from_function_response(name=call.name, response={"result": result})],
                ))

## 5. Score answers and export results

In [ ]:
results_list = []
maximum_attempts = 20
llm_model = "gemini-2.5-flash"
output_file = "scored_answers.csv"

for position, (index, row) in enumerate(df.iterrows(), start=1):
    print(f"[{position}/{len(df)}] Row processing...")
    for question_column in df.columns:
        if question_column not in rules_dict:
            logger.warning("Question '%s' is not defined in the JSON rules. Skipping.", question_column)
            continue

        scoring_method, range_min, range_max = rules_dict[question_column]
        answer_text = row.get(question_column, "")
        if pd.isna(answer_text) or not str(answer_text).strip():
            logger.warning("Row %d, question '%s' has no answer. Skipping.", position, question_column)
            continue

        safe_answer_text = security.main_security_sanitization(str(answer_text))
        final_evaluation = agent_loop(
            llm_model=llm_model,
            current_question=question_column,
            rules=scoring_method,
            range_min=range_min,
            range_max=range_max,
            answer_text=safe_answer_text,
            maximum_attempts=maximum_attempts,
        )

        if final_evaluation is None:
            logger.error("Evaluation failed for row %d, question '%s'.", position, question_column)
            continue

        try:
            eval_data = json.loads(final_evaluation)
        except json.JSONDecodeError:
            logger.error("Invalid JSON response for row %d, question '%s'.", position, question_column)
            results_list.append({
                "respondent_row": position,
                "question": question_column,
                "original_answer": safe_answer_text,
                "raw_evaluation": final_evaluation,
            })
            continue

        results_list.append({
            "respondent_row": position,
            "question": question_column,
            "original_answer": safe_answer_text,
            "score": eval_data.get("score"),
            "confidence": eval_data.get("confidence"),
            "reasoning": eval_data.get("reasoning"),
        })

result_df = pd.DataFrame(results_list)
result_df.to_csv(output_file, index=False)
logger.info("Results saved to '%s'", output_file)
result_df.head()